# Priority Global Human-Impact Watershed Exports

This notebook creates global watershed tables for **LandScan population, GHSL residential population, Global Dam Watch barriers, NPKGRIDS fertilizer, and HydroWASTE wastewater**. It uses the same watershed boundaries, site choices, Google Drive exports, and task timing as the ERA5-Land notebook.

LandScan is exported once each year from 2000-2024. GHSL is exported every five years from 1975-2020; its 2025 and 2030 future estimates are not included. Dams, fertilizer, and wastewater are each exported once because they are one-time maps.

In [ ]:
REPO_URL = "https://github.com/global-river-chem/data-workflow_spatial.git"
REPO_DIR = "/content/data-workflow_spatial"

import os
import subprocess
import sys


if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-r", "requirements.txt"], check=True)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

In [ ]:
from src.gee_spatial.auth import initialize
from src.gee_spatial.extraction import load_run_config
from src.gee_spatial.human_impacts import (
    available_dataset_years,
    extract_human_impact_dataset,
    human_impact_export_columns,
    inspect_human_impact_assets,
    load_human_impact_config,
    validate_asset_report,
)

assets = load_run_config("config/gee-assets.yml")
human_impacts = load_human_impact_config("config/human-impact-products.yml")

DRIVE_EXPORT_FOLDER = "Google-Earth-Engine"
assets.setdefault("exports", {}).update(
    {"destination": "drive", "drive_folder": DRIVE_EXPORT_FOLDER}
)
ee = initialize(project=assets.get("earth_engine", {}).get("project"))

print("Earth Engine project:", assets.get("earth_engine", {}).get("project"))
print("Watershed data source:", assets["watersheds"]["asset_id"])
print("Drive output folder:", assets["exports"].get("drive_folder"))

## Edit Run Settings

You only need to edit the next cell. The defaults use all watersheds and all five datasets. Set `MAX_TASKS_TO_LAUNCH = 5` for a small test run: it launches one task for each dataset. `FERTILIZER_CROPS = []` uses all crop types.

In [ ]:
from src.gee_spatial.extraction import load_watersheds
from src.gee_spatial.runs import choose_property_name

RUN_LABEL = "all_sites"
DATASETS = ["population", "ghsl_population", "dams", "fertilizer", "wastewater"]

LANDSCAN_START_YEAR = 2000
LANDSCAN_END_YEAR = 2024
LANDSCAN_YEARS_TO_SKIP = []
GHSL_EPOCHS_TO_SKIP = []
STATIC_DATASETS_TO_SKIP = []

# Set to 5 for a small test run with one task per dataset.
MAX_TASKS_TO_LAUNCH = None

# Leave empty to include all crops.
FERTILIZER_CROPS = []

# Site options: "all_sites", "lter_list", or "site_id_list".
SITE_FILTER_MODE = "all_sites"
LTER_FILTERS = []
SITE_IDS = []

valid_datasets = set(human_impacts["datasets"])
unknown_datasets = sorted(set(DATASETS) - valid_datasets)
if unknown_datasets:
    raise ValueError(f"Unknown DATASETS: {unknown_datasets}")

available_landscan_years = available_dataset_years(human_impacts, "population")
landscan_years = [
    year
    for year in range(LANDSCAN_START_YEAR, LANDSCAN_END_YEAR + 1)
    if year not in set(LANDSCAN_YEARS_TO_SKIP)
]
invalid_landscan_years = sorted(set(landscan_years) - set(available_landscan_years))
if "population" in DATASETS and invalid_landscan_years:
    raise ValueError(f"LandScan years are outside the configured range: {invalid_landscan_years}")

ghsl_years = [
    year
    for year in available_dataset_years(human_impacts, "ghsl_population")
    if year not in set(GHSL_EPOCHS_TO_SKIP)
]

watersheds = load_watersheds(assets["watersheds"]["asset_id"])
property_names = watersheds.first().propertyNames().getInfo()
site_id_column = choose_property_name(watersheds, "site_id")
lter_column = choose_property_name(watersheds, "lter", alternatives=["LTER"])
lter_values = watersheds.aggregate_array(lter_column).distinct().sort().getInfo()
lter_lookup = {str(value).strip().lower(): value for value in lter_values}

if SITE_FILTER_MODE == "all_sites":
    selected_watersheds = watersheds
elif SITE_FILTER_MODE == "lter_list":
    if not LTER_FILTERS:
        raise ValueError('Set LTER_FILTERS when SITE_FILTER_MODE = "lter_list".')
    selected_lter_values = [
        lter_lookup.get(str(value).strip().lower(), value)
        for value in LTER_FILTERS
    ]
    selected_watersheds = watersheds.filter(ee.Filter.inList(lter_column, selected_lter_values))
elif SITE_FILTER_MODE == "site_id_list":
    if not SITE_IDS:
        raise ValueError('Set SITE_IDS when SITE_FILTER_MODE = "site_id_list".')
    selected_watersheds = watersheds.filter(ee.Filter.inList(site_id_column, SITE_IDS))
else:
    raise ValueError('SITE_FILTER_MODE must be "all_sites", "lter_list", or "site_id_list".')

selected_row_count = selected_watersheds.size().getInfo()
if selected_row_count == 0:
    raise ValueError("No watershed rows were selected.")

print("Watershed rows in source:", watersheds.size().getInfo())
print("Selected watershed rows:", selected_row_count)
print("Watershed fields:", property_names)
print("Run label:", RUN_LABEL)
print("Datasets:", DATASETS)
print("LandScan years:", landscan_years if "population" in DATASETS else [])
print("GHSL years:", ghsl_years if "ghsl_population" in DATASETS else [])
print("One-time datasets skipped:", STATIC_DATASETS_TO_SKIP)
print("Fertilizer crops:", FERTILIZER_CROPS or "all")
print("Selected preview:")
print(selected_watersheds.limit(5).getInfo())

In [ ]:
from pprint import pprint

asset_report = inspect_human_impact_assets(human_impacts)
validate_asset_report(human_impacts, asset_report)
pprint(asset_report)
print("Source-data check passed.")

In [ ]:
import csv
import time
from pathlib import Path

from src.gee_spatial.export import export_table, task_summary
from src.gee_spatial.timing import (
    datetime_label,
    task_timing_row,
    timing_rows_from_launched_tasks,
    utc_now,
    write_timing_log,
)

static_datasets = [
    dataset_name
    for dataset_name in DATASETS
    if human_impacts["datasets"][dataset_name]["temporal_resolution"] == "static"
    and dataset_name not in set(STATIC_DATASETS_TO_SKIP)
]
run_plan = [
    {"dataset": dataset_name, "year": None}
    for dataset_name in static_datasets
]
dynamic_years = {
    "population": landscan_years,
    "ghsl_population": ghsl_years,
}
dynamic_datasets = [
    dataset_name for dataset_name in DATASETS if dataset_name in dynamic_years
]
# Put the newest year or date from each changing dataset near the front
# so the five-task test checks every dataset with recent values.
run_plan.extend(
    {"dataset": dataset_name, "year": dynamic_years[dataset_name][-1]}
    for dataset_name in dynamic_datasets
    if dynamic_years[dataset_name]
)
run_plan.extend(
    {"dataset": dataset_name, "year": year}
    for dataset_name in dynamic_datasets
    for year in dynamic_years[dataset_name][:-1]
)
if MAX_TASKS_TO_LAUNCH is not None:
    run_plan = run_plan[: int(MAX_TASKS_TO_LAUNCH)]
if not run_plan:
    raise ValueError("The current settings produce no export tasks.")

run_name = f"human_impacts_priority_{RUN_LABEL}"
run_wall_clock_started_at = utc_now()
run_wall_clock_timer_started = time.perf_counter()
session_started_at = run_wall_clock_started_at
timing_log_path = Path("timing-logs") / (
    f"gee_task_timing_human_impacts_{RUN_LABEL}_{datetime_label(session_started_at)}.csv"
)
run_timing_summary_path = Path("timing-logs") / (
    f"gee_run_wall_clock_human_impacts_{RUN_LABEL}_{datetime_label(session_started_at)}.csv"
)
timing_log_path.parent.mkdir(exist_ok=True)


def iso_or_blank(value):
    return value.isoformat(timespec="seconds") if value else ""


def write_run_wall_clock_summary(status, finished_at=None, elapsed_min=None):
    if elapsed_min is None and finished_at is not None:
        elapsed_min = (finished_at - run_wall_clock_started_at).total_seconds() / 60
    elapsed_hr = elapsed_min / 60 if elapsed_min is not None else ""
    row = {
        "run_name": run_name,
        "run_label": RUN_LABEL,
        "status": status,
        "task_plan": ";".join(
            f"{item['dataset']}:{item['year'] or 'static'}" for item in run_plan
        ),
        "n_tasks": len(run_plan),
        "selected_rows": selected_row_count,
        "site_filter_mode": SITE_FILTER_MODE,
        "lter_filters": ";".join(str(value) for value in LTER_FILTERS),
        "site_id_filters": ";".join(str(value) for value in SITE_IDS),
        "datasets": ";".join(DATASETS),
        "fertilizer_crops": "all" if not FERTILIZER_CROPS else ";".join(FERTILIZER_CROPS),
        "started_at_utc": iso_or_blank(run_wall_clock_started_at),
        "finished_at_utc": iso_or_blank(finished_at),
        "elapsed_min": "" if elapsed_min is None else f"{elapsed_min:.3f}",
        "elapsed_hr": "" if elapsed_hr == "" else f"{elapsed_hr:.3f}",
        "task_timing_log": str(timing_log_path),
        "drive_export_folder": assets["exports"].get("drive_folder", ""),
    }
    with run_timing_summary_path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(row))
        writer.writeheader()
        writer.writerow(row)
    return row


launched_tasks = []
print("Run wall-clock start:", run_wall_clock_started_at.isoformat(timespec="seconds"))
print("Task plan:", run_plan)
print("Selected watershed rows:", selected_row_count)
print("Drive output folder:", assets["exports"].get("drive_folder"))

for item in run_plan:
    dataset_name = item["dataset"]
    year = item["year"]
    period_label = str(year) if year is not None else "static"
    out_name = f"human_impacts_{dataset_name}_{period_label}_{RUN_LABEL}_watershed_extract"
    selectors = human_impact_export_columns(human_impacts, dataset_name)
    print()
    print("Launching:", out_name)
    print("Export columns:", selectors)

    export_rows = extract_human_impact_dataset(
        dataset_name,
        human_impacts,
        selected_watersheds,
        year=year,
        fertilizer_crops=FERTILIZER_CROPS or None,
    )
    task = export_table(
        export_rows,
        description=out_name,
        export_config=assets["exports"],
        file_name_prefix=out_name,
        selectors=selectors,
    )

    launched_at = utc_now()
    run_row = {
        "run_name": run_name,
        "mode": "human_impacts",
        "products": [dataset_name],
        "year": year,
        "month": None,
        "months": None,
        "period": human_impacts["datasets"][dataset_name]["temporal_resolution"],
        "run_group": RUN_LABEL,
    }
    timing_row = task_timing_row(
        run_row=run_row,
        export_name=out_name,
        task=task,
        selected_rows=selected_row_count,
        export_destination=assets["exports"].get("destination", "drive"),
        launched_at=launched_at,
    )
    launched_tasks.append(
        {"name": out_name, "task": task, "launched_at": launched_at, "timing_row": timing_row}
    )
    write_timing_log(timing_rows_from_launched_tasks(launched_tasks), timing_log_path)
    print(task_summary(task))

write_run_wall_clock_summary(status="launched")
print()
print("Launched tasks:", len(launched_tasks))
print("Task timing log:", timing_log_path)
print("Run timing summary:", run_timing_summary_path)

In [ ]:
import time

from src.gee_spatial.timing import (
    DONE_STATES,
    print_timing_summary,
    timing_rows_from_launched_tasks,
    update_task_timing_row,
    utc_now,
    write_timing_log,
)

poll_seconds = 60
if "launched_tasks" not in globals() or not launched_tasks:
    print("No tasks were launched in this session.")
else:
    while True:
        now = utc_now()
        print()
        print("Task status at", now.isoformat(timespec="seconds"))
        all_done = True
        for item in launched_tasks:
            item["timing_row"] = update_task_timing_row(
                item["timing_row"], item["task"], checked_at=now
            )
            state = item["timing_row"].get("state")
            elapsed_min = float(item["timing_row"].get("elapsed_min") or 0)
            print(f"{item['name']}: {state}, elapsed {elapsed_min:.1f} min")
            if state not in DONE_STATES:
                all_done = False

        write_timing_log(timing_rows_from_launched_tasks(launched_tasks), timing_log_path)
        if all_done:
            break
        time.sleep(poll_seconds)

    run_wall_clock_finished_at = utc_now()
    run_wall_clock_elapsed_min = (time.perf_counter() - run_wall_clock_timer_started) / 60
    print_timing_summary(timing_rows_from_launched_tasks(launched_tasks))
    print()
    print("Run wall-clock finish:", run_wall_clock_finished_at.isoformat(timespec="seconds"))
    print(f"Run wall-clock elapsed: {run_wall_clock_elapsed_min:.1f} min ({run_wall_clock_elapsed_min / 60:.2f} hr)")
    write_run_wall_clock_summary(
        status="completed",
        finished_at=run_wall_clock_finished_at,
        elapsed_min=run_wall_clock_elapsed_min,
    )
    print("Task timing log:", timing_log_path)
    print("Run timing summary:", run_timing_summary_path)

## What the values mean

- Population totals use the watershed's average population density, its area, and the share of the watershed with population data. For very small watersheds, the notebook retries at a finer scale without counting the same population cell twice.
- Dams and wastewater are one-time maps. HydroWASTE uses the location supplied with the dataset and includes plants marked `Operational`, `Construction Completed`, or `Not Reported`.
- Fertilizer values compare the summed crop-specific application rates within a watershed. They are not a total fertilizer mass because the source does not say how much area each crop occupies.

After Earth Engine finishes, organize the CSVs from `Google-Earth-Engine` into a run-specific Drive folder. The raw filenames begin with `human_impacts_`.